# PyRosetta Scoring Functions

![PyRosetta Scoring Functions](https://proto-bio.github.io/proto-assets/images/tool/pyrosetta/hero.png)

This notebook demonstrates the five PyRosetta operations wrapped in `proto_tools`: Spatial Aggregation Propensity (SAP), Solvent Accessible Surface Area (SASA), Rosetta energy scoring, FastRelax (returns a relaxed `Structure`), and interface analysis for two-chain complexes. Each tool takes one or more protein structures and returns physics-based metrics with per-residue breakdowns where applicable.

**License:** PyRosetta requires acceptance of the [Rosetta Software License](https://www.rosettacommons.org/software/license-and-download). Free for academic and non-commercial use; commercial users must obtain a separate license.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("pyrosetta")
display_overview("pyrosetta")
display_docs_section("pyrosetta", "Background")

# PyRosetta Scoring Functions

PyRosetta provides physics-based scoring of protein structures using the Rosetta molecular modeling suite. Five operations are available: Spatial Aggregation Propensity (SAP) scoring, Solvent Accessible Surface Area (SASA) computation, Rosetta energy scoring, standalone FastRelax that returns the relaxed structure, and interface analysis of two-chain complexes via `InterfaceAnalyzerMover`. The three scoring tools and the interface analyzer all expose an opt-in `pre_relax_structures` preprocess that calls `pyrosetta-relax` before scoring.

## Background

**Spatial Aggregation Propensity (SAP)** quantifies how much hydrophobic surface area is exposed on a protein. Proteins with large patches of exposed hydrophobicity are prone to [aggregation](https://en.wikipedia.org/wiki/Protein_aggregation) -- a major concern in therapeutic protein development. SAP computes a per-atom hydrophobicity score weighted by solvent exposure, then sums across the surface. Higher SAP scores indicate greater aggregation risk.

**Solvent Accessible Surface Area (SASA)** measures the total surface area of a protein that is accessible to solvent molecules (modeled as a spherical probe, typically 1.4 A radius for water). SASA is fundamental to understanding protein folding thermodynamics: buried residues contribute to the [hydrophobic core](https://en.wikipedia.org/wiki/Hydrophobic_core), while exposed residues interact with solvent and binding partners. Per-residue SASA values reveal which positions are buried vs. solvent-exposed.

**Rosetta Energy Scoring** evaluates protein structures using a physics-based [energy function](https://en.wikipedia.org/wiki/Force_field_(chemistry)) that combines van der Waals interactions, electrostatics, hydrogen bonding, solvation, and backbone torsion preferences. The score is reported in Rosetta Energy Units (REU). Lower (more negative) total energies indicate more favorable structures. [FastRelax](https://www.rosettacommons.org/docs/latest/scripting_documentation/RosettaScripts/Movers/movers_pages/FastRelaxMover) is optionally applied before scoring to resolve steric clashes and backbone strain.

**Interface Analysis** quantifies the quality of protein-protein interfaces using Rosetta's `InterfaceAnalyzerMover` plus auxiliary residue-composition and surface-layer analyses. Seven metrics are emitted per complex: shape complementarity (`interface_sc`, 0-1), interface hydrogen-bond count (`interface_hbonds`), binding ΔG (`interface_dG`, in REU), buried SASA (`interface_dSASA`, Å²), interface packing statistic (`interface_packstat`, 0-1), interface-residue apolar fraction (`interface_hydrophobicity`, %), and binder surface apolar fraction (`surface_hydrophobicity`, 0-1).

## Available tools

In [2]:
display_available_tools("pyrosetta")

- **`run_pyrosetta_energy()`** — Compute Rosetta energy scores for protein structures (with optional FastRelax preprocess via config.pre_relax_structures)
- **`run_pyrosetta_interface_analyzer()`** — Compute interface-quality metrics for a two-chain complex via Rosetta's InterfaceAnalyzerMover + LayerSelector (with optional FastRelax preprocess via config.pre_relax_structures)
- **`run_pyrosetta_relax()`** — Run PyRosetta FastRelax on a structure and return the relaxed Structure plus its total score
- **`run_pyrosetta_sap()`** — Compute Spatial Aggregation Propensity (SAP) scores for protein structures using PyRosetta
- **`run_pyrosetta_sasa()`** — Compute Solvent Accessible Surface Area (SASA) for protein structures using PyRosetta

### `run_pyrosetta_relax`

FastRelax minimizes a structure under the chosen score function and returns the relaxed coordinates as a `Structure`. Use this when downstream filters or scorers need to operate on a relaxed pose (e.g. cofolding filter pipelines for binder design). The relaxed `Structure` chains directly into the other PyRosetta scorers or into geometric `Structure` methods.


In [3]:
from proto_tools import (
    PyRosettaEnergyConfig,
    PyRosettaEnergyInput,
    PyRosettaRelaxConfig,
    PyRosettaRelaxInput,
    run_pyrosetta_energy,
    run_pyrosetta_relax,
)
from proto_tools.tools.structure_scoring.pyrosetta.pyrosetta_relax import example_input as relax_example_input


In [4]:
# Display input docs
display_api_reference("pyrosetta", "input", "run_pyrosetta_relax")

relax_inputs = relax_example_input()


**Input** — `PyRosettaRelaxInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `inputs` | `list[proto_tools.tools.structure_scoring.pyrosetta.shared_data_models.ScoringStructureInput]` | required | Protein structures to relax |

In [5]:
# Display config docs
display_api_reference("pyrosetta", "config", "run_pyrosetta_relax")

# Single FastRelax cycle to match Germinal's `-relax:default_repeats 1` and keep the example fast
relax_config = PyRosettaRelaxConfig(scorefxn="ref2015", relax_cycles=1)


**Config** — `PyRosettaRelaxConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cpu'` | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| `timeout` | `int` | `600` | Maximum execution time in seconds |
| `seed` | `int \| None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `scorefxn` | `str` | `'ref2015'` | Rosetta score function name |
| `relax_cycles` | `int` | `1` | Number of FastRelax repeats (more = better convergence, slower) |
| `constrain_to_start` | `bool` | `True` | Constrain relaxation to starting coordinates |
| `max_iter` | `int \| None` | `None` | Maximum minimizer iterations per relax cycle. None uses PyRosetta default (2500). |
| `disable_jumps` | `bool` | `False` | Lock inter-chain rigid-body DOFs during relaxation. |
| `min_type` | `str \| None` | `None` | Optional FastRelax minimizer type. |
| `align_to_start` | `bool` | `False` | Align the relaxed pose back onto the starting pose after FastRelax. |
| `copy_b_factors_from_start` | `bool` | `False` | Copy per-residue B-factors from the starting pose to the relaxed pose. |

In [6]:
# Run FastRelax
relax_result = run_pyrosetta_relax(relax_inputs, relax_config)


Running run_pyrosetta_relax [00:00]

> NOTE: The 3D viewer below renders locally (JupyterLab, VS Code) but not in GitHub previews.

In [7]:
# Display output docs and inspect results
display_api_reference("pyrosetta", "output", "run_pyrosetta_relax")

r = relax_result.results[0]
print(f"Total score (relaxed): {r.total_score:.1f} REU")
print(f"Cycles applied:        {r.relax.relax_cycles}")
print(f"Output chains:         {r.relax.relaxed_structure.get_chain_ids()}")

# The relaxed Structure is ready to chain into another scorer.
relaxed_structure = r.relax.relaxed_structure
relaxed_structure.visualize(color_by="chain")


**Output** — `PyRosettaRelaxOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `results` | `list[proto_tools.tools.structure_scoring.pyrosetta.pyrosetta_relax.PyRosettaRelaxMetrics]` | `[]` | Relax results, one per input structure |

Total score (relaxed): -140.5 REU
Cycles applied:        1
Output chains:         ['A']


In [8]:
# Chain the relaxed Structure into pyrosetta-energy. Default config = no further relax.
chained = run_pyrosetta_energy(
    PyRosettaEnergyInput(inputs=[relaxed_structure]),
)
print(f"Re-scored relaxed pose: {chained.results[0].total_energy:.1f} REU")


Re-scored relaxed pose: -140.5 REU


### `run_pyrosetta_sap`

SAP (Spatial Aggregation Propensity) quantifies how much hydrophobic surface area is exposed on a protein. Proteins with large patches of exposed hydrophobicity are prone to aggregation — a major concern in therapeutic protein development. `run_pyrosetta_sap` loads each input structure into a Rosetta Pose and scores it with Rosetta's `core.pack.guidance_scoreterms.sap` module, returning a single scalar SAP score per structure. Higher SAP scores indicate greater aggregation risk.

In [9]:
from proto_tools import (
    PyRosettaSAPConfig,
    PyRosettaSAPInput,
    run_pyrosetta_sap,
)
from proto_tools.tools.structure_scoring.pyrosetta.pyrosetta_sap import example_input as sap_example_input

In [10]:
# Display input docs
display_api_reference("pyrosetta", "input", "run_pyrosetta_sap")

# Use the tool's canonical example input PDB.
sap_inputs = sap_example_input()

# Let's take a look at what this looks like
sap_inputs.inputs[0].structure.visualize(color_by="chain")

**Input** — `PyRosettaSAPInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `inputs` | `list[proto_tools.tools.structure_scoring.pyrosetta.shared_data_models.ScoringStructureInput]` | required | Protein structures with optional chain selection for SAP scoring |

In [11]:
# Display config docs
display_api_reference("pyrosetta", "config", "run_pyrosetta_sap")

# PyRosettaSAPConfig has no tool-specific parameters; all defaults are used
sap_config = PyRosettaSAPConfig()

**Config** — `PyRosettaSAPConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cpu'` | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| `timeout` | `int` | `600` | Maximum execution time in seconds |
| `seed` | `int \| None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `pre_relax_structures` | `bool` | `False` | If True, run pyrosetta-relax on each input structure before scoring. |
| `relax_config` | `proto_tools.tools.structure_scoring.pyrosetta.pyrosetta_relax.PyRosettaRelaxConfig` | `PyRosettaRelaxConfig(verbose=0, device='cpu', timeout=600, seed=None, scorefxn='ref2015', relax_cycles=1, constrain_to_start=True, max_iter=None, disable_jumps=False, min_type=None, align_to_start=False, copy_b_factors_from_start=False)` | Settings used when pre_relax_structures=True. Ignored otherwise. |

In [12]:
# Run the SAP scoring function
sap_result = run_pyrosetta_sap(sap_inputs, sap_config)

In [13]:
# Display output docs and inspect results
display_api_reference("pyrosetta", "output", "run_pyrosetta_sap")

for r in sap_result.results:
    print(f"SAP score: {r.sap_score:.2f}")

**Output** — `PyRosettaSAPOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `results` | `list[proto_tools.tools.structure_scoring.pyrosetta.pyrosetta_sap.PyRosettaSAPMetrics]` | `[]` | SAP scores with per-residue breakdown, one per input structure |

SAP score: 42.08


### `run_pyrosetta_sasa`

SASA (Solvent Accessible Surface Area) measures the total protein surface accessible to a spherical solvent probe (1.4 A by default, matching water). Buried residues contribute to the hydrophobic core, while exposed residues interact with solvent and binding partners. `run_pyrosetta_sasa` returns both the total SASA and a per-residue breakdown, which is useful for identifying surface-exposed sites, characterizing the hydrophobic core, or feeding into downstream feature engineering.

In [14]:
from proto_tools import (
    PyRosettaSASAConfig,
    PyRosettaSASAInput,
    run_pyrosetta_sasa,
)
from proto_tools.tools.structure_scoring.pyrosetta.pyrosetta_sasa import example_input as sasa_example_input

In [15]:
# Display input docs
display_api_reference("pyrosetta", "input", "run_pyrosetta_sasa")

sasa_inputs = sasa_example_input()

**Input** — `PyRosettaSASAInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `inputs` | `list[proto_tools.tools.structure_scoring.pyrosetta.shared_data_models.ScoringStructureInput]` | required | Protein structures with optional chain selection for SASA computation |

In [16]:
# Display config docs
display_api_reference("pyrosetta", "config", "run_pyrosetta_sasa")

# 1.4 A is the standard water probe radius
sasa_config = PyRosettaSASAConfig(probe_radius=1.4)

**Config** — `PyRosettaSASAConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cpu'` | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| `timeout` | `int` | `600` | Maximum execution time in seconds |
| `seed` | `int \| None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `probe_radius` | `float` | `1.4` | Solvent probe radius in Angstroms (standard water = 1.4) |
| `pre_relax_structures` | `bool` | `False` | If True, run pyrosetta-relax on each input structure before scoring. |
| `relax_config` | `proto_tools.tools.structure_scoring.pyrosetta.pyrosetta_relax.PyRosettaRelaxConfig` | `PyRosettaRelaxConfig(verbose=0, device='cpu', timeout=600, seed=None, scorefxn='ref2015', relax_cycles=1, constrain_to_start=True, max_iter=None, disable_jumps=False, min_type=None, align_to_start=False, copy_b_factors_from_start=False)` | Settings used when pre_relax_structures=True. Ignored otherwise. |

In [17]:
# Run the SASA computation
sasa_result = run_pyrosetta_sasa(sasa_inputs, sasa_config)

In [18]:
# Display output docs
display_api_reference("pyrosetta", "output", "run_pyrosetta_sasa")

# Total SASA plus the five most solvent-exposed residues
r = sasa_result.results[0]
print(f"Total SASA: {r.total_sasa:.1f} A^2")
print(f"Residues:   {len(r.per_residue)}")

top_exposed = sorted(r.per_residue, key=lambda res: res.sasa, reverse=True)[:5]
print("\nTop 5 most exposed residues:")
for res in top_exposed:
    print(f"  {res.chain_id}:{res.residue_name}{res.residue_index} — {res.sasa:.1f} A^2")

**Output** — `PyRosettaSASAOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `results` | `list[proto_tools.tools.structure_scoring.pyrosetta.pyrosetta_sasa.PyRosettaSASAMetrics]` | `[]` | SASA results, one per input structure |

Total SASA: 4978.9 A^2
Residues:   65

Top 5 most exposed residues:
  A:LYS46 — 235.9 A^2
  A:ARG21 — 214.8 A^2
  A:MET1 — 195.9 A^2
  A:PRO65 — 186.1 A^2
  A:LYS35 — 179.9 A^2


### `run_pyrosetta_energy`

Rosetta energy scoring evaluates a protein structure with a physics-based energy function that combines van der Waals interactions, electrostatics, hydrogen bonding, solvation, and backbone torsion terms. The score is reported in Rosetta Energy Units (REU); lower (more negative) is more favorable. By default, `run_pyrosetta_energy` scores the input structure as-given; setting `pre_relax_structures=True` on the config opts into the [FastRelax](https://www.rosettacommons.org/docs/latest/scripting_documentation/RosettaScripts/Movers/movers_pages/FastRelaxMover) preprocess (composing `pyrosetta-relax` under the hood) to resolve clashes and backbone strain first. Returns the total energy, a per-term breakdown, and per-residue totals. The default score function is `ref2015`, Rosetta's current production all-atom energy function.


In [19]:
from proto_tools import (
    PyRosettaEnergyConfig,
    PyRosettaEnergyInput,
    PyRosettaRelaxConfig,
    run_pyrosetta_energy,
)
from proto_tools.tools.structure_scoring.pyrosetta.pyrosetta_energy import example_input as energy_example_input


In [20]:
# Display input docs
display_api_reference("pyrosetta", "input", "run_pyrosetta_energy")

energy_inputs = energy_example_input()

**Input** — `PyRosettaEnergyInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `inputs` | `list[proto_tools.tools.structure_scoring.pyrosetta.shared_data_models.ScoringStructureInput]` | required | Protein structures with optional chain selection for energy scoring |

In [21]:
# Display config docs
display_api_reference("pyrosetta", "config", "run_pyrosetta_energy")

# Use ref2015 with the FastRelax preprocess (1 cycle to keep the example fast)
energy_config = PyRosettaEnergyConfig(
    scorefxn="ref2015",
    pre_relax_structures=True,
    relax_config=PyRosettaRelaxConfig(relax_cycles=1),
)


**Config** — `PyRosettaEnergyConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cpu'` | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| `timeout` | `int` | `600` | Maximum execution time in seconds |
| `seed` | `int \| None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `scorefxn` | `str` | `'ref2015'` | Rosetta score function name |
| `pre_relax_structures` | `bool` | `False` | If True, run pyrosetta-relax on each input structure before scoring. |
| `relax_config` | `proto_tools.tools.structure_scoring.pyrosetta.pyrosetta_relax.PyRosettaRelaxConfig` | `PyRosettaRelaxConfig(verbose=0, device='cpu', timeout=600, seed=None, scorefxn='ref2015', relax_cycles=1, constrain_to_start=True, max_iter=None, disable_jumps=False, min_type=None, align_to_start=False, copy_b_factors_from_start=False)` | Settings used when pre_relax_structures=True. Ignored otherwise. |

In [22]:
# Run the energy scoring function
energy_result = run_pyrosetta_energy(energy_inputs, energy_config)

In [23]:
# Display output docs
display_api_reference("pyrosetta", "output", "run_pyrosetta_energy")

r = energy_result.results[0]
print(f"Total energy: {r.total_energy:.1f} REU")
print("\nTop contributing energy terms:")
for term, value in sorted(r.energy_terms.items(), key=lambda kv: kv[1])[:6]:
    print(f"  {term:>12s}: {value:.2f}")


**Output** — `PyRosettaEnergyOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `results` | `list[proto_tools.tools.structure_scoring.pyrosetta.pyrosetta_energy.PyRosettaEnergyMetrics]` | `[]` | Energy scores, one per input structure |

Total energy: -140.6 REU

Top contributing energy terms:
        fa_atr: -334.32
       fa_elec: -125.94
       p_aa_pp: -30.58
   hbond_sr_bb: -25.53
   hbond_lr_bb: -18.35
   hbond_bb_sc: -6.87


### `run_pyrosetta_interface_analyzer`

`InterfaceAnalyzerMover` + `LayerSelector` produce a suite of interface-quality metrics for complexes: shape complementarity, interface H-bond count, binding ΔG, buried SASA, interface packing, and interface/surface hydrophobicity. The interface is identified by the `target_chains` / `binder_chain` fields on each `InterfaceStructureInput` (alongside the structure itself) — chain labels are validated against the structure at construction time.

In [24]:
from proto_tools import (
    PyRosettaInterfaceAnalyzerConfig,
    PyRosettaInterfaceAnalyzerInput,
    run_pyrosetta_interface_analyzer,
)
from proto_tools.tools.structure_scoring.pyrosetta.pyrosetta_interface_analyzer import (
    example_input as interface_example_input,
)


In [25]:
# Display input docs
display_api_reference("pyrosetta", "input", "run_pyrosetta_interface_analyzer")

interface_inputs = interface_example_input()


**Input** — `PyRosettaInterfaceAnalyzerInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `inputs` | `list[proto_tools.tools.structure_scoring.pyrosetta.pyrosetta_interface_analyzer.InterfaceStructureInput]` | required | Complexes to analyze, each with target_chains/binder_chain labels |

In [26]:
# Display config docs
display_api_reference("pyrosetta", "config", "run_pyrosetta_interface_analyzer")

# Defaults: scorefxn="ref2015", no FastRelax preprocess.
interface_config = PyRosettaInterfaceAnalyzerConfig()


**Config** — `PyRosettaInterfaceAnalyzerConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cpu'` | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| `timeout` | `int` | `600` | Maximum execution time in seconds |
| `seed` | `int \| None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `scorefxn` | `str` | `'ref2015'` | Rosetta score function name |
| `pre_relax_structures` | `bool` | `False` | If True, run pyrosetta-relax on each input structure before analyzing. |
| `relax_config` | `proto_tools.tools.structure_scoring.pyrosetta.pyrosetta_relax.PyRosettaRelaxConfig` | `PyRosettaRelaxConfig(verbose=0, device='cpu', timeout=600, seed=None, scorefxn='ref2015', relax_cycles=1, constrain_to_start=True, max_iter=None, disable_jumps=False, min_type=None, align_to_start=False, copy_b_factors_from_start=False)` | Settings used when pre_relax_structures=True. Ignored otherwise. |

In [27]:
# Run the interface analyzer
interface_result = run_pyrosetta_interface_analyzer(interface_inputs, interface_config)


In [28]:
# Display output docs
display_api_reference("pyrosetta", "output", "run_pyrosetta_interface_analyzer")

m = interface_result.results[0]
print(f"interface_sc:                 {m.interface_sc:.3f}")
print(f"interface_hbonds:             {m.interface_hbonds}")
print(f"interface_dG:                 {m.interface_dG:.2f} REU")
print(f"interface_dSASA:              {m.interface_dSASA:.1f} Å²")
print(f"interface_packstat:           {m.interface_packstat:.3f}")
print(f"interface_hydrophobicity:     {m.interface_hydrophobicity:.1f} %")
print(f"surface_hydrophobicity:       {m.surface_hydrophobicity:.3f}")


**Output** — `PyRosettaInterfaceAnalyzerOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `results` | `list[proto_tools.tools.structure_scoring.pyrosetta.pyrosetta_interface_analyzer.PyRosettaInterfaceAnalyzerMetrics]` | `[]` | Interface-analysis metrics, one per input structure |

interface_sc:                 0.649
interface_hbonds:             2
interface_dG:                 181.52 REU
interface_dSASA:              1204.8 Å²
interface_packstat:           0.680
interface_hydrophobicity:     42.9 %
surface_hydrophobicity:       0.431


## Export Results

Each output model supports export to common file formats for downstream analysis. SASA results export well to CSV because of the per-residue table; energy results export to JSON to preserve the nested per-term breakdown.

In [29]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

sap_result.export("pyrosetta_sap", export_path=output_dir, file_format="json")
sasa_result.export("pyrosetta_sasa", export_path=output_dir, file_format="csv")
energy_result.export("pyrosetta_energy", export_path=output_dir, file_format="json")
interface_result.export("pyrosetta_interface", export_path=output_dir, file_format="csv")

for name in (
    "pyrosetta_sap.json",
    "pyrosetta_sasa.csv",
    "pyrosetta_energy.json",
    "pyrosetta_interface.csv",
):
    print(f"Exported {output_dir / name}")

Exported example_output/pyrosetta_sap.json
Exported example_output/pyrosetta_sasa.csv
Exported example_output/pyrosetta_energy.json
Exported example_output/pyrosetta_interface.csv
